# Packages import


In [ ]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from webdriver_manager.chrome import ChromeDriverManager

import json
import os

# Ceneo scraper

1. Provide url adress of product's opinion page

In [ ]:
product_code="124893467"
page=1
url=f"https://www.ceneo.pl/{product_code}/opinie-{page}"
print(url)

https://www.ceneo.pl/124893467/opinie-1


2. Send the request to provided url adress

In [ ]:
path_to_driver  = ChromeDriverManager().install()
service = Service(path_to_driver)
driver = webdriver.Chrome(service=service)
driver.get(url)
driver.find_element(by='xpath',value='//*[@id="js_cookie-consent-general"]/div/div[2]/button[1]').click()

In [ ]:
response = requests.get(url=url)

3. If status code is OK, fetch all opinions from requested webpage

In [ ]:
page_dom = BeautifulSoup(response.text,'html.parser')
soup = page_dom.find('h1',{'class':'product-topproduct-infoname'}).get_text()

In [ ]:
# print(page_dom.find_all('div',{'class':'user-post__card'}))
all_opinions = page_dom.find_all('div',{'class':'js_product-review'})

In [ ]:
opinions = [opinion for opinion in page_dom.find_all('div',{'class':'js_product-review'}) if "user-post--highlight" not in  opinion.get("class",[])]
print(len(opinions))
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(len(opinions))

4. For all fetched opinions, parse them to extract relevant data 


In [ ]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        "opinion_id":opinion["data-entry-id"],
        "author":opinion.select_one("span.user-post__author-name").get_text().strip(),
        'reccomendation':opinion.select_one('span.user-post__author-recommendation>em').get_text().strip() if opinion.
            select_one('span.user-post__author-recommendation>em') else None,
        "score":opinion.select_one('.user-post__score-count').get_text().strip(),
        "content":opinion.select_one('div.user-post__text').get_text().strip(),
        "pros":[p.get_text().strip() for p in opinion.select('div.review-feature__item--positive')],
        "cons":[c.get_text().strip() for c in opinion.select('div.review-feature__item--negative')],
        "like":opinion.select_one('button.vote-yes > span').get_text().strip(),
        "dislike":opinion.select_one('button.vote-no > span').get_text().strip(),
        "publishing_date":opinion.select_one('span.user-post__published > time:nth-child(1)')["datetime"].strip() if opinion.select_one('span.user-post__published > time:nth-child(1)') else None,
        "purchase_date":opinion.select_one('span.user-post__published > time:nth-child(2)')['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)') else None,
    }
    all_opinions.append(single_opinion)
    
json.dump(all_opinions,open("all_opinions.json",'w'),indent=4)

6. Check if there is next page

In [ ]:
next = True if page_dom.select_one("button.pagination__next") else False
if next: page+=1

[<div class="user-post user-post__card js_product-review" data-entry-id="17657913">
 <div class="user-post__header">
 <div class="js_lazy user-post__avatar user-rank__avatar" data-bg="/Content/img/account/avatar/0.svg"></div>
 </div>
 <div class="user-post__body">
 <div class="user-post__content">
 <div>
 <span class="user-post__author-name">
 l...z
 </span>
 <span class="user-post__score">
 <span class="screen-reader-text">Ocena:</span>
 <span class="score-container score-container--s js_score-container">
 <span class="score-marker score-marker--s" style="width: 100.0%"></span>
 </span>
 <span class="user-post__score-count">5/5</span>
 <span class="user-post__author-recomendation">
 <em class="recommended">Polecam</em>
 </span>
 </span>
 </div>
 <div class="mb-4">
 <span class="verified-purchase">
 Zaufana Opinia potwierdzona zakupem
 </span>
 <span class="user-post__published">
 <span class="user-post__published">
 Wystawiono
 <time datetime="2023-06-28 22:50:27">3 lata temu, </time>

8. Save obtained opinions

In [ ]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [ ]:
with open(f"./opinions/{product_code}.json",'w',encoding="UTF-8") as file:
    json.dump(all_opinions,file,indent=4)

FileNotFoundError: [Errno 2] No such file or directory: "./opinions/124893467.json,'w',encoding='UTF-8'"